# Category C – Demo & Test Prediction [Group 22]

This notebook demonstrates how to generate predictions for the Natural Language Inference (NLI) task using the trained transformer-based models.

---

## Approach

The system uses a **hybrid ensemble of two RoBERTa-based models**:

- **Cross-Encoder**
  - Processes the premise and hypothesis jointly
  - Captures detailed token-level interactions

- **Bi-Encoder**
  - Encodes sentences independently
  - Captures overall semantic similarity

---

## Ensemble Strategy

The final prediction is obtained by combining both models using a weighted average:

final_logits = α · cross_logits + (1 - α) · bi_logits

where \(alpha) is chosen based on development set performance.

- This allows the model to balance detailed interaction (cross-encoder) and general semantic similarity (bi-encoder)

---

## Demo notebook

- Loads trained cross-encoder and bi-encoder models  
- Loads tokenizer and metadata  
- Preprocesses test data  
- Runs inference to obtain logits from both models  
- Applies the ensemble strategy to compute final predictions  
- Saves predictions in the required submission format  

---


In [ ]:
# imports and setup

import os
import json
import random
from collections import Counter
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    RobertaModel,
    DataCollatorWithPadding,
)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
BASE_DIR = os.getcwd()

OUT_DIR = os.path.join(BASE_DIR, "outputs", "category_C")

CROSS_DIR = os.path.join(OUT_DIR, "cross_encoder")
BI_DIR = os.path.join(OUT_DIR, "bi_encoder")
ENSEMBLE_DIR = os.path.join(OUT_DIR, "ensemble"
)
SUBMISSION_DIR = os.path.join(OUT_DIR, "submissions")

# Create all folders
os.makedirs(CROSS_DIR, exist_ok=True)
os.makedirs(BI_DIR, exist_ok=True)
os.makedirs(ENSEMBLE_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print("Output directory:", OUT_DIR)

In [ ]:
TEST_PATH = "data/test.csv"

MODEL_NAME = "roberta-base"
OUTPUT_DIR = CROSS_DIR
BI_OUTPUT_DIR = BI_DIR
ENSEMBLE_OUTPUT_DIR = ENSEMBLE_DIR

assert os.path.exists(TEST_PATH), f"Test file not found: {TEST_PATH}"
assert os.path.exists(os.path.join(OUTPUT_DIR, "cross_encoder_model.pt")), "Cross-encoder checkpoint not found."
assert os.path.exists(os.path.join(OUTPUT_DIR, "cross_encoder_metadata.json")), "Cross metadata not found."
assert os.path.exists(os.path.join(BI_OUTPUT_DIR, "bi_encoder_model.pt")), "Bi-encoder checkpoint not found."

print("Test file:", TEST_PATH)
print("Cross model dir:", OUTPUT_DIR)
print("Bi model dir:", BI_OUTPUT_DIR)
print("Ensemble dir:", ENSEMBLE_OUTPUT_DIR)

In [ ]:
# load metadata and test data

with open(os.path.join(OUTPUT_DIR, "cross_encoder_metadata.json"), "r") as f:
    cross_metadata = json.load(f)

test_df = pd.read_csv(TEST_PATH)

print("Test shape:", test_df.shape)
display(test_df.head())

MAX_LENGTH = cross_metadata["max_length"]
BATCH_SIZE = cross_metadata["batch_size"]
DROPOUT = cross_metadata["dropout"]
HIDDEN_DIM_1 = cross_metadata["hidden_dim_1"]
HIDDEN_DIM_2 = cross_metadata["hidden_dim_2"]

### Test Data Preprocessing

This section detects the correct columns for premise and hypothesis automatically.

It cleans the text by:
- Removing missing values
- Stripping whitespace
- Filtering out empty rows

This ensures clean and valid input for the models.

In [ ]:
def guess_column(df: pd.DataFrame, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

premise_col = guess_column(test_df, ["premise", "sentence1", "text1", "s1"])
hypothesis_col = guess_column(test_df, ["hypothesis", "sentence2", "text2", "s2"])
label_col = guess_column(test_df, ["label", "gold_label", "class", "y"])

if premise_col is None or hypothesis_col is None:
    raise ValueError(f"Could not detect premise/hypothesis columns. Found: {list(test_df.columns)}")

test_df[premise_col] = test_df[premise_col].fillna("").astype(str).str.strip()
test_df[hypothesis_col] = test_df[hypothesis_col].fillna("").astype(str).str.strip()

# keep only rows with non-empty text
test_df = test_df[(test_df[premise_col] != "") & (test_df[hypothesis_col] != "")].copy()
test_df = test_df.reset_index(drop=True)

print("Detected columns:")
print("premise_col:", premise_col)
print("hypothesis_col:", hypothesis_col)
print("label_col:", label_col)

print("Cleaned test shape:", test_df.shape)

In [ ]:
# label mapping

has_labels = label_col is not None

if has_labels:
    unique_labels = sorted(test_df[label_col].unique().tolist())
    label2id = {lab: i for i, lab in enumerate(unique_labels)}
    id2label = {i: lab for lab, i in label2id.items()}

    test_df["label_id"] = test_df[label_col].map(label2id)
    if test_df["label_id"].isna().sum() != 0:
        raise ValueError("Some test labels could not be mapped.")
    test_df["label_id"] = test_df["label_id"].astype(int)
    num_labels = len(label2id)

    print("Labels found in test file.")
    print("label2id:", label2id)
else:
    # fallback to metadata label mapping if test has no labels
    if "label2id" in cross_metadata:
        raw_label2id = cross_metadata["label2id"]
        label2id = {int(k) if str(k).isdigit() else k: v for k, v in raw_label2id.items()} if isinstance(list(raw_label2id.keys())[0], str) else raw_label2id
        num_labels = len(label2id)
    else:
        num_labels = cross_metadata["num_labels"]

    print("No labels found in test file. Running prediction-only mode.")
    print("num_labels:", num_labels)

In [ ]:
# Load tokenizer from model name for stability

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", MODEL_NAME)

In [ ]:
# dataset classes

class NLIPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, max_length: int, label_col: str = None):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.max_length = max_length
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]

        enc = self.tokenizer(
            prem,
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        if self.label_col is not None:
            enc["labels"] = int(row[self.label_col])

        return enc


class NLIBiEncoderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, max_length: int, label_col: str = None):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.max_length = max_length
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]

        prem_enc = self.tokenizer(
            prem,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        hyp_enc = self.tokenizer(
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        item = {
            "premise_input_ids": prem_enc["input_ids"],
            "premise_attention_mask": prem_enc["attention_mask"],
            "hypothesis_input_ids": hyp_enc["input_ids"],
            "hypothesis_attention_mask": hyp_enc["attention_mask"],
        }

        if self.label_col is not None:
            item["labels"] = int(row[self.label_col])

        return item

In [ ]:
# dataloaders

def bi_collate_fn(features):
    premise_features = [
        {
            "input_ids": f["premise_input_ids"],
            "attention_mask": f["premise_attention_mask"]
        }
        for f in features
    ]

    hypothesis_features = [
        {
            "input_ids": f["hypothesis_input_ids"],
            "attention_mask": f["hypothesis_attention_mask"]
        }
        for f in features
    ]

    premise_batch = tokenizer.pad(
        premise_features,
        padding=True,
        return_tensors="pt"
    )

    hypothesis_batch = tokenizer.pad(
        hypothesis_features,
        padding=True,
        return_tensors="pt"
    )

    batch = {
        "premise_input_ids": premise_batch["input_ids"],
        "premise_attention_mask": premise_batch["attention_mask"],
        "hypothesis_input_ids": hypothesis_batch["input_ids"],
        "hypothesis_attention_mask": hypothesis_batch["attention_mask"],
    }

    if "labels" in features[0]:
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)

    return batch


pair_label_col = "label_id" if has_labels else None
bi_label_col = "label_id" if has_labels else None

test_ds = NLIPairDataset(
    test_df,
    tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    max_length=MAX_LENGTH,
    label_col=pair_label_col
)

test_bi_ds = NLIBiEncoderDataset(
    test_df,
    tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    max_length=MAX_LENGTH,
    label_col=bi_label_col
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=data_collator
)

test_bi_loader = DataLoader(
    test_bi_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=bi_collate_fn
)

print("Test batches (cross):", len(test_loader))
print("Test batches (bi):", len(test_bi_loader))

In [ ]:
# model definitions

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class RobertaCustomNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_repr)
        return logits


class RobertaBiEncoderNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        combined_size = hidden_size * 3

        self.classifier = nn.Sequential(
            nn.Linear(combined_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(
        self,
        premise_input_ids,
        premise_attention_mask,
        hypothesis_input_ids,
        hypothesis_attention_mask
    ):
        premise_outputs = self.encoder(
            input_ids=premise_input_ids,
            attention_mask=premise_attention_mask
        )
        hypothesis_outputs = self.encoder(
            input_ids=hypothesis_input_ids,
            attention_mask=hypothesis_attention_mask
        )

        u = mean_pooling(premise_outputs.last_hidden_state, premise_attention_mask)
        v = mean_pooling(hypothesis_outputs.last_hidden_state, hypothesis_attention_mask)

        features = torch.cat([u, v, torch.abs(u - v)], dim=-1)
        logits = self.classifier(features)
        return logits

In [ ]:
# load trained models

cross_ckpt = os.path.join(OUTPUT_DIR, "cross_encoder_model.pt")
bi_ckpt = os.path.join(BI_OUTPUT_DIR, "bi_encoder_model.pt")

print("Cross checkpoint exists:", os.path.exists(cross_ckpt))
print("Bi checkpoint exists:", os.path.exists(bi_ckpt))
print("Cross checkpoint size:", os.path.getsize(cross_ckpt))
print("Bi checkpoint size:", os.path.getsize(bi_ckpt))

best_model = RobertaCustomNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)
best_model.load_state_dict(torch.load(cross_ckpt, map_location=device))
best_model.to(device)
best_model.eval()

best_bi_model = RobertaBiEncoderNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)
best_bi_model.load_state_dict(torch.load(bi_ckpt, map_location=device))
best_bi_model.to(device)
best_bi_model.eval()

print("Both models loaded successfully.")

In [ ]:
# get logits from both models

@torch.no_grad()
def get_cross_logits(model, dataloader):
    model.eval()
    all_logits = []

    for batch in tqdm(dataloader, desc="Cross logits", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(logits.cpu())

    return torch.cat(all_logits, dim=0)


@torch.no_grad()
def get_bi_logits(model, dataloader):
    model.eval()
    all_logits = []

    for batch in tqdm(dataloader, desc="Bi logits", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        all_logits.append(logits.cpu())

    return torch.cat(all_logits, dim=0)


cross_logits = get_cross_logits(best_model, test_loader)
bi_logits = get_bi_logits(best_bi_model, test_bi_loader)

print("Cross logits shape:", cross_logits.shape)
print("Bi logits shape:", bi_logits.shape)

In [ ]:
# load ensemble weight

ensemble_meta_path = os.path.join(ENSEMBLE_OUTPUT_DIR, "ensemble_metadata.json")

with open(ensemble_meta_path, "r") as f:
    ensemble_metadata = json.load(f)

alpha = float(ensemble_metadata["best_alpha"])
print("Using alpha:", alpha)

In [ ]:
# final ensemble predictions

final_logits = alpha * cross_logits + (1.0 - alpha) * bi_logits
final_preds = torch.argmax(final_logits, dim=-1).numpy()

print("Number of predictions:", len(final_preds))
print("Prediction sample:", final_preds[:20])

In [ ]:
submission_path = os.path.join(SUBMISSION_DIR, "Group_22_C.csv")

# Create dataframe with correct column name
submission_df = pd.DataFrame({"prediction": final_preds})

# Save as CSV
submission_df.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission_df.head())